# Handwritten Digit Recognition with a CNN
**Module ITLPA701 - Python and Fundamentals of AI - CAT2 Practical 2026**

Workflow: *Apply Data Preprocessing -> Apply Deep Learning Algorithms -> Apply Model Evaluation Techniques -> Demonstration -> Responsible Use*.

Run the cells from top to bottom (Kernel -> Restart & Run All). Nothing is hard-coded: every number printed below is computed by this notebook.
This notebook does **not** overwrite `models/digit_cnn_model.keras`; use `python train.py` for the saved model.

## 1. Imports and environment

In [ ]:
import os, sys, copy
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))   # make config.py and src/ importable

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf, keras
import config as C
from src.data_loader import load_digits_dataset, dataset_statistics, get_splits
from src.preprocessing import prepare_inputs, augment_with_shifts, digit_array_to_pil
from src.model import build_cnn
from src.training import train_one, diagnose_history, score_on_split
from src.evaluation import predict_classes, compute_metrics, interpret_metrics, most_confused_pairs
from src.utils import set_global_seed
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

set_global_seed(C.SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", C.SEED)

## 2. Dataset loading (data acquisition)
The scikit-learn Digits dataset is bundled with scikit-learn - no download needed.

In [ ]:
images, labels = load_digits_dataset()
print("images:", images.shape, "labels:", labels.shape)

## 3. Dataset inspection

In [ ]:
stats = dataset_statistics(images, labels)
for k, v in stats.items():
    print(f"{k}: {v}")

## 4. Visualisation

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for d in range(10):
    idx = np.where(labels == d)[0][:2]
    for r in range(2):
        axes[r, d].imshow(images[idx[r]], cmap="gray_r"); axes[r, d].axis("off")
        if r == 0: axes[r, d].set_title(f"digit {d}")
plt.suptitle("Sample images (8x8)"); plt.show()

counts = np.bincount(labels)
plt.figure(figsize=(6, 3)); plt.bar(range(10), counts); plt.xticks(range(10))
plt.title("Class distribution"); plt.xlabel("digit"); plt.ylabel("images"); plt.show()

## 5. Preprocessing and feature engineering
1. **Split first** (stratified, fixed seed) so no information from the test set influences preprocessing.
2. **Normalise** pixels 0..16 -> 0..1 with the fixed constant 16 (input scaling; no statistics are learned from data).
3. **Reshape** `(N, 8, 8)` -> `(N, 8, 8, 1)` - the channel axis Conv2D expects.
4. **Labels** stay integers 0-9 and use `sparse_categorical_crossentropy` (equivalent to one-hot + categorical cross-entropy).
5. **Augmentation** (random +-1 px shifts) is applied to the training split only, used in the adjusted model.

*Feature engineering for a CNN:* raw pixels are the input representation; normalisation and reshaping prepare it; the convolutional layers then **learn** features (edges -> strokes -> curves/loops). These learned feature maps are the model-level features (shown in section 8b).

In [ ]:
splits = get_splits(C.SEED)
print("train", splits["X_train"].shape, "| validation", splits["X_val"].shape, "| test", splits["X_test"].shape)
print("value range after normalisation:", splits["X_train"].min(), "-", splits["X_train"].max())
Xa, ya = augment_with_shifts(splits["X_train"], splits["y_train"], seed=C.SEED)
print("training set with shift augmentation:", Xa.shape)

## 6. CNN architecture and model summary

In [ ]:
model = build_cnn(C.ADJUSTED_CONFIG)
model.summary()

**Why these parameters?**
Conv 32 and 64 filters with 3x3 kernels, `same` padding (8x8 images are tiny, so borders must not shrink them); one 2x2 max-pool (8x8 -> 4x4); Dense 64; Dropout 0.30 (regularisation); softmax over 10 classes; Adam with learning rate 0.001 (robust default); sparse categorical cross-entropy (integer labels, 10 classes); batch size 32 (small dataset, many weight updates per epoch).

## 7. Training: baseline model
The baseline reproduces the original proposal: 10 epochs, no dropout, no augmentation, no callbacks.

In [ ]:
base_cfg = copy.deepcopy(C.BASELINE_CONFIG)
base_model, base_hist = train_one(base_cfg, splits, verbose=0)
print("epochs run:", len(base_hist["loss"]))

## 8. Training graphs and diagnosis of the baseline

In [ ]:
def plot_hist(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for a, key in zip(ax, ("accuracy", "loss")):
        a.plot(h[key], label="training"); a.plot(h["val_" + key], label="validation")
        a.set_title(f"{title}: {key}"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
    plt.show()

plot_hist(base_hist, "Baseline")
base_diag = diagnose_history(base_hist)
print("\n".join("- " + f for f in base_diag["findings"]))

### 8b. What the convolutional layer learned (model-level features)

In [ ]:
feat_model = keras.Model(base_model.input, base_model.get_layer("conv1").output)
sample = splits["X_test"][:1]
maps = feat_model.predict(sample, verbose=0)[0]
fig, axes = plt.subplots(2, 9, figsize=(14, 3.2))
axes[0, 0].imshow(sample[0, :, :, 0], cmap="gray_r"); axes[0, 0].set_title("input")
for i in range(1, 18):
    ax = axes.flat[i]; ax.imshow(maps[:, :, i - 1], cmap="viridis")
for ax in axes.flat: ax.axis("off")
plt.suptitle("Input image and the first 17 feature maps of Conv layer 1"); plt.show()

## 9. Evaluation-based adjustment: train the adjusted model
Changes: dropout 0.0 -> 0.30, +-1 px shift augmentation, up to 60 epochs with EarlyStopping (patience 8, best weights restored), ReduceLROnPlateau.

In [ ]:
adj_cfg = copy.deepcopy(C.ADJUSTED_CONFIG)
adj_model, adj_hist = train_one(adj_cfg, splits, verbose=0)
plot_hist(adj_hist, "Adjusted")
adj_diag = diagnose_history(adj_hist)
print("epochs run:", adj_diag["epochs_run"], "| best validation-loss epoch:", adj_diag["best_val_loss_epoch"])
print("\n".join("- " + f for f in adj_diag["findings"]))

## 10. Evaluation: accuracy, precision, recall, F1
Validation data chooses the model; the **test** set is only used for reporting.

In [ ]:
import pandas as pd
rows = []
for name, m in (("baseline", base_model), ("adjusted", adj_model)):
    for split in ("val", "test"):
        s = score_on_split(m, splits[f"X_{split}"], splits[f"y_{split}"])
        rows.append({"model": name, "split": split, "accuracy": s["accuracy"],
                     "precision(macro)": s["precision_macro"], "recall(macro)": s["recall_macro"],
                     "F1(macro)": s["f1_macro"], "loss": s["loss"]})
comparison = pd.DataFrame(rows).round(4)
comparison

In [ ]:
val = comparison[comparison.split == "val"].set_index("model")
final_name = "adjusted" if (val.loc["adjusted", "accuracy"], -val.loc["adjusted", "loss"]) >= \
                           (val.loc["baseline", "accuracy"], -val.loc["baseline", "loss"]) else "baseline"
final_model = adj_model if final_name == "adjusted" else base_model
print("Selected on validation data:", final_name)

## 11. Classification report and confusion matrix (final model, test set)

In [ ]:
y_pred, probs = predict_classes(final_model, splits["X_test"])
y_true = splits["y_test"]
print(classification_report(y_true, y_pred, digits=4))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6.5, 5.5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title(f"Confusion matrix - {final_name} model"); plt.show()

## 12. Interpretation (generated from the real numbers)

In [ ]:
print(interpret_metrics(compute_metrics(y_true, y_pred) | {"most_confused_pairs":
      [{"true": t, "predicted": p, "count": c} for t, p, c in most_confused_pairs(cm)]}))

## 13. Sample predictions

In [ ]:
rng = np.random.default_rng(C.SEED)
pick = rng.choice(len(y_true), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(12, 5.2))
for ax, i in zip(axes.flat, pick):
    ax.imshow(splits["X_test"][i, :, :, 0], cmap="gray_r"); ax.axis("off")
    ok = y_pred[i] == y_true[i]
    ax.set_title(f"true {y_true[i]} | pred {y_pred[i]}\n{probs[i].max():.0%}", color="green" if ok else "red")
plt.show()

## 14. Uploaded-image inference (same pipeline as the Streamlit app)

In [ ]:
from predict import predict_image
demo = digit_array_to_pil(splits["X_test_raw"][pick[0]])   # a dataset digit rendered as a 256x256 black-on-white image
res = predict_image(final_model, demo)
print("Predicted digit:", res["digit"], "| confidence:", f"{res['confidence']:.2%}", "| true:", y_true[pick[0]])
fig, ax = plt.subplots(1, 2, figsize=(6, 3)); ax[0].imshow(demo, cmap="gray"); ax[1].imshow(res["preview"], cmap="gray_r")
ax[0].set_title("uploaded"); ax[1].set_title("8x8 model input"); [a.axis("off") for a in ax]; plt.show()

## 15. Limitations and responsible use
* Only 1,797 small 8x8 images from a limited set of writers: unusual handwriting, blur, tilt, shadows or poor contrast can cause errors.
* Uploaded photos must be shrunk to 8x8, so cropping/inversion mistakes change the prediction.
* A high confidence is not a guarantee of correctness.

**Mitigation:** clear image instructions, a confidence threshold with a warning and second-best guess, a preview of the 8x8 model input, and mandatory human verification where accuracy matters.